**Parse JSON, clean, deduplicate → Silver table**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Define exact schema matching your JSON structure
json_schema = StructType([
    StructField("invoice", StructType([
        StructField("client_name",    StringType()),
        StructField("client_address", StringType()),
        StructField("seller_name",    StringType()),
        StructField("seller_address", StringType()),
        StructField("invoice_number", StringType()),
        StructField("invoice_date",   StringType()),
        StructField("due_date",       StringType()),
    ])),
    StructField("items", ArrayType(StructType([
        StructField("description", StringType()),
        StructField("quantity",    StringType()),
        StructField("total_price", StringType()),
    ]))),
    StructField("subtotal", StructType([
        StructField("tax",      StringType()),
        StructField("discount", StringType()),
        StructField("total",    StringType()),
    ])),
    StructField("payment_instructions", StructType([
        StructField("due_date",        StringType()),
        StructField("bank_name",       StringType()),
        StructField("account_number",  StringType()),
        StructField("payment_method",  StringType()),
    ]))
])

print("Schema defined.")

Step 2 — Parse JSON and Flatten All Fields


In [0]:
# incremental processing in batch daily runs
# option 1

# from pyspark.sql.functions import current_date

# # Only process today's Bronze partition — partition pruning kicks in
# df_bronze_incremental = (
#     spark.table("invoices.bronze.raw_invoices")
#     .filter(col("ingestion_date") == current_date())   # reads ONLY today's partition folder
# )



# # option 2:
#     # ── SETUP (run once) ─────────────────────────────────────────
# spark.sql("""
#     CREATE TABLE IF NOT EXISTS invoices.silver.pipeline_watermark (
#         pipeline_name   STRING,
#         last_processed  TIMESTAMP
#     )
#     USING DELTA
# """)

# # Insert initial watermark if table is empty
# spark.sql("""
#     INSERT INTO invoices.silver.pipeline_watermark
#     SELECT 'silver_transformation', TIMESTAMP '2000-01-01 00:00:00'
#     WHERE NOT EXISTS (
#         SELECT 1 FROM invoices.silver.pipeline_watermark
#         WHERE pipeline_name = 'silver_transformation'
#     )
# """)

# # ── ON EVERY RUN ─────────────────────────────────────────────

# # Step 1 — Read last watermark
# last_processed = spark.sql("""
#     SELECT last_processed 
#     FROM invoices.silver.pipeline_watermark
#     WHERE pipeline_name = 'silver_transformation'
# """).first()[0]

# print(f"Last processed timestamp: {last_processed}")

# # Step 2 — Read only NEW Bronze rows since last watermark
# df_bronze_incremental = (
#     spark.table("invoices.bronze.raw_invoices")
#     .filter(col("ingested_at") > last_processed)
# )

# new_rows = df_bronze_incremental.count()
# print(f"New rows to process: {new_rows}")

# # Step 3 — Process and write to Silver (run your parse + explode logic here)
# # ... your transformation code ...

# # Step 4 — Update watermark ONLY after successful Silver write
# spark.sql("""
#     UPDATE invoices.silver.pipeline_watermark
#     SET last_processed = current_timestamp()
#     WHERE pipeline_name = 'silver_transformation'
# """)

# print("Watermark updated.")


In [0]:
from pyspark.sql.functions import try_to_date, regexp_replace

df_parsed = (
    spark.table("invoices.bronze.raw_invoices")
    .filter(col("json_data").isNotNull())

    # Parse the JSON string into a struct column
    .withColumn("parsed", from_json(col("json_data"), json_schema))

    # Flatten invoice fields
    .withColumn("client_name",      col("parsed.invoice.client_name"))
    .withColumn("client_address",   col("parsed.invoice.client_address"))
    .withColumn("seller_name",      col("parsed.invoice.seller_name"))
    .withColumn("seller_address",   col("parsed.invoice.seller_address"))
    .withColumn("invoice_number",   col("parsed.invoice.invoice_number"))
    .withColumn("invoice_date",     try_to_date(col("parsed.invoice.invoice_date"), "MM/dd/yyyy"))
    .withColumn("invoice_due_date", try_to_date(col("parsed.invoice.due_date"),     "MM/dd/yyyy"))

    # Flatten subtotal fields — clean numeric strings (remove spaces, commas, %, then try_cast to DOUBLE)
    .withColumn("tax",              regexp_replace(regexp_replace(regexp_replace(col("parsed.subtotal.tax"), " ", ""), ",", "."), "%", "").try_cast(DoubleType()))
    .withColumn("discount",         regexp_replace(regexp_replace(regexp_replace(col("parsed.subtotal.discount"), " ", ""), ",", "."), "%", "").try_cast(DoubleType()))
    .withColumn("total",            regexp_replace(regexp_replace(regexp_replace(col("parsed.subtotal.total"), " ", ""), ",", "."), "%", "").try_cast(DoubleType()))

    # Flatten payment instruction fields
    .withColumn("payment_due_date", try_to_date(col("parsed.payment_instructions.due_date"), "MM/dd/yyyy"))
    .withColumn("bank_name",        col("parsed.payment_instructions.bank_name"))
    .withColumn("account_number",   col("parsed.payment_instructions.account_number"))
    .withColumn("payment_method",   col("parsed.payment_instructions.payment_method"))

    # Keep items as array for now — will explode below
    .withColumn("items",            col("parsed.items"))

    # Drop raw JSON and intermediate parsed struct
    .drop("parsed", "json_data")
)

print(f"Rows after parse: {df_parsed.count()}")
display(df_parsed.limit(3))

Step 3 — Explode Items Array (One Row Per Line Item)

In [0]:
# Each invoice has multiple items — explode creates one row per item
df_silver = (
    df_parsed
    .withColumn("item", explode_outer(col("items")))   # explode_outer keeps invoices with no items
    .withColumn("item_description", col("item.description"))
    .withColumn("item_quantity",    regexp_replace(regexp_replace(col("item.quantity"), " ", ""), ",", ".").try_cast(DoubleType()))
    .withColumn("item_total_price", regexp_replace(regexp_replace(col("item.total_price"), " ", ""), ",", ".").try_cast(DoubleType()))
    .drop("items", "item")                             # drop array and struct after extraction
    # Add Silver metadata
    .withColumn("pipeline_layer",   lit("silver"))
    .withColumn("cleaned_at",       current_timestamp())
)

print(f"Rows after explode (one per line item): {df_silver.count()}")
df_silver.printSchema()
display(df_silver.limit(5))

In [0]:
# Quick profile before writing
from pyspark.sql.functions import count, when

total = df_silver.count()

display(
    df_silver.select(
        count(when(col("invoice_number").isNull(),   1)).alias("null_invoice_number"),
        count(when(col("client_name").isNull(),      1)).alias("null_client_name"),
        count(when(col("total").isNull(),            1)).alias("null_total"),
        count(when(col("invoice_date").isNull(),     1)).alias("null_invoice_date"),
        count(when(col("item_total_price").isNull(), 1)).alias("null_item_price"),
        count(when(col("item_quantity").isNull(),    1)).alias("null_item_qty"),
    )
)

# Check duplicates — same invoice_number + item_description
dup_count = (
    df_silver
    .groupBy("invoice_number", "item_description")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Total rows          : {total}")
print(f"Duplicate line items: {dup_count}")

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS invoices.silver.pipeline_watermark (
        pipeline_name   STRING,
        last_processed  TIMESTAMP
    )
    USING DELTA
""")

spark.sql("""
    INSERT INTO invoices.silver.pipeline_watermark
    SELECT 'silver_transformation', TIMESTAMP '2000-01-01 00:00:00'
    WHERE NOT EXISTS (
        SELECT 1 FROM invoices.silver.pipeline_watermark
        WHERE pipeline_name = 'silver_transformation'
    )
""")

print("Watermark table ready.")
spark.sql("SELECT * FROM invoices.silver.pipeline_watermark").show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Keep latest record per invoice_number + item_description
window_spec = Window.partitionBy("invoice_number", "item_description") \
                    .orderBy(col("ingested_at").desc())

df_silver_deduped = (
    df_silver
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f"Rows before dedup : 5602")
print(f"Rows after dedup  : {df_silver_deduped.count()}")

In [0]:
(
    df_silver_deduped
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("ingestion_date")
    .saveAsTable("invoices.silver.invoices_clean")
)

print(f"Silver table written!")
print(f"Total rows in Silver: {spark.table('invoices.silver.invoices_clean').count()}")

In [0]:
# Only update watermark AFTER Silver write succeeds
# If Silver write had failed above, watermark stays at old value
# so next run re-processes the same batch — idempotent behaviour

spark.sql("""
    UPDATE invoices.silver.pipeline_watermark
    SET last_processed = current_timestamp()
    WHERE pipeline_name = 'silver_transformation'
""")

print("Watermark updated.")
spark.sql("SELECT * FROM invoices.silver.pipeline_watermark").show()

Why update watermark AFTER write and not before?
If the Silver write fails midway, the watermark stays unchanged — so the next run re-processes the same batch. This gives you at-least-once guarantee and prevents data loss. This is the standard pattern in production pipelines.